In [ ]:
!pip install bayesian-optimization
!pip install scikit-optimize
!pip install hyperopt
!pip install ConfigSpace
!pip install smac
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
from bayes_opt import BayesianOptimization
from bayes_opt.acquisition import ExpectedImprovement
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from skopt import forest_minimize
from skopt.space import Real
from ConfigSpace import ConfigurationSpace, Float
from smac import HyperparameterOptimizationFacade, Scenario

## Patient response function and Bayesian optimzer algorithm functions.

In [ ]:
import pandas as pd
import numpy as np
from bayes_opt import BayesianOptimization
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from ConfigSpace import ConfigurationSpace, Float
from smac import HyperparameterOptimizationFacade, Scenario

# Default response-surface family: 2 optima, baseline smoothness, no added measurement noise
# Amplitude dose-response uses an asymmetric biphasic Hill curve (see bi_hill_response below),
# not a symmetric Gaussian, per Goutelle et al. 2008 (Hill eq. in pharmacology) and
# Cedergreen, Ritz & Streibig 2005 (biphasic/hormetic dose-response models).
baseline_family = {
    'name': 'baseline_2optima',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'biphasic_hill',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'

}

three_optima_family = {
    'name': '3optima',
    'num_optima': 3,
    'spread_scale': 1.0,
    'function_shape': 'biphasic_hill',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

broad_peaks_family = {
    'name': 'broad_optima',
    'num_optima': 2,
    'spread_scale': 2.0,
    'function_shape': 'biphasic_hill',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

sharp_peaks_family = {
    'name': 'sharp_optima',
    'num_optima': 2,
    'spread_scale': 0.4,
    'function_shape': 'biphasic_hill',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

heteroscedastic_noise_family = {
    'name': 'heteroscedastic_noise',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'biphasic_hill',
    'noise': 'heteroscedastic',
    'unique_parameter_interaction': 'none'
}

heavy_tailed_noise_family = {
    'name': 'heavy_tailed_noise',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'biphasic_hill',
    'noise': 'heavy_tailed',
    'unique_parameter_interaction': 'none'
}

unique_parameter_interaction_family = {
    'name': 'unique_parameter_interaction',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'biphasic_hill',
    'noise': 'normal',
    'unique_parameter_interaction': 'pulsewidth-amplitude'
}



def bi_hill_response(amplitude, ec50: float, ic50: float, hill_n_rise: float = 4.0, hill_n_fall: float = 4.0):
    """Asymmetric biphasic Hill response."""
    amplitude = np.maximum(amplitude, 0.001)

    rising_component = (
        amplitude**hill_n_rise
        / (amplitude**hill_n_rise + ec50**hill_n_rise)
    )

    falling_component = (
        ic50**hill_n_fall
        / (amplitude**hill_n_fall + ic50**hill_n_fall)
    )

    response = rising_component * falling_component

    # Numerically normalize the curve so its maximum is 1.
    amplitude_grid = np.linspace(0.001, 10.0, 10000)

    rising_grid = (
        amplitude_grid**hill_n_rise
        / (amplitude_grid**hill_n_rise + ec50**hill_n_rise)
    )

    falling_grid = (
        ic50**hill_n_fall
        / (amplitude_grid**hill_n_fall + ic50**hill_n_fall)
    )

    maximum_response = np.max(rising_grid * falling_grid)

    return response / maximum_response

def generate_patient_profile(patient_seed: int, family: dict):
    """Draws one unique synthetic patient's response landscape for a given surface family, independent of any optimizer's own RNG."""
    rng = np.random.default_rng(patient_seed)

    base_optima = [(10.0 + rng.uniform(-2.0, 2.0), 500.0 + rng.uniform(-25.0, 25.0)), 
                   (25.0 + rng.uniform(-2.0, 2.0), 100.0 + rng.uniform(-25.0, 25.0))] #may need to change this later on.
    num_optima = family['num_optima']
    optima = []
    for numOfOptima in range(num_optima):
        if numOfOptima < len(base_optima):
            opt_f, opt_w = base_optima[numOfOptima]
        else:
            opt_f = rng.uniform(5.0, 50.0)
            opt_w = rng.uniform(50.0, 600.0)
        optima.append((opt_f, opt_w))

    spread_scale = family['spread_scale']

    return {
        'patient_seed': patient_seed,
        'surface_family': family['name'],
        'ideal_amp_pup': rng.normal(4.4, 0.15),
        # Asymmetric spreads: faster rise toward threshold, slower decline above it
        'amp_rise_pup': rng.uniform(0.25, 0.35),
        'amp_fall_pup': rng.uniform(0.55, 0.75),
        'ideal_amp_hrv': rng.normal(4.0, 0.15),
        'amp_rise_hrv': rng.uniform(0.2, 0.3),
        'amp_fall_hrv': rng.uniform(0.45, 0.65),
        'optima': optima,
        'freq_spread': rng.uniform(4.0, 6.0) * spread_scale,
        'width_spread': rng.uniform(80.0, 120.0) * spread_scale,
    }


def run_optimization_comparison(num_simulation_trials: int, amount_of_patients: int, patient_and_machine_startSeed : int, patient_type: dict):

    search_boundaries = {
        'amplitude': (0.5, 5.0),       # mA
        'frequency': (1.0, 50.0),      # Hz
        'pulse_width': (50.0, 500.0),  # microseconds
    }
    
    all_patients_results = []  # accumulates each patient's comparison_data across the loop

    for i in range(amount_of_patients):

        patient_profile = generate_patient_profile(patient_and_machine_startSeed, patient_type)
        ideal_amp_pup = patient_profile['ideal_amp_pup']
        amp_rise_pup = patient_profile['amp_rise_pup']
        amp_fall_pup = patient_profile['amp_fall_pup']
        ideal_amp_hrv = patient_profile['ideal_amp_hrv']
        amp_rise_hrv = patient_profile['amp_rise_hrv']
        amp_fall_hrv = patient_profile['amp_fall_hrv']
        optima = patient_profile['optima']
        freq_spread = patient_profile['freq_spread']
        width_spread = patient_profile['width_spread']

        def realistic_patient_curve(amplitude, frequency, pulse_width, AddNoise, rng = None):

            if not AddNoise:
                # Pupillometry Calculation (asymmetric biphasic Hill dose-response, see bi_hill_response)
                amp_comp_pup = bi_hill_response(amplitude, ideal_amp_pup - amp_rise_pup, ideal_amp_pup + amp_fall_pup)
                amp_comp_hrv = bi_hill_response(amplitude, ideal_amp_hrv - amp_rise_hrv, ideal_amp_hrv + amp_fall_hrv)

                pup_score = 100 * amp_comp_pup 
                hrv_score = 100 * amp_comp_hrv 
                w_pup, w_hrv = 0.5, 0.5
                amplitude_score = w_pup * pup_score + w_hrv * hrv_score

                # We calculate response based on proximity to the optimal (Freq, Width) pairs
                def get_interdependence_score(f, w, Amplitude_Scale_Factor = 1.0):
                    scores = []
                    if Amplitude_Scale_Factor != 1.0:
                        for (opt_f, opt_w) in optima:
                            dist = (((f - opt_f)**2 / (2 * freq_spread**2))* Amplitude_Scale_Factor +
                                    ((w - opt_w)**2 / (2 * width_spread**2)))
                            scores.append(np.exp(-dist))
                    else:
                        for (opt_f, opt_w) in optima:
                            dist = (((f - opt_f)**2 / (2 * freq_spread**2)) +
                                    ((w - opt_w)**2 / (2 * width_spread**2)))
                            scores.append(np.exp(-dist))

                    return max(scores)

                if patient_type.get('unique_parameter_interaction') == 'pulsewidth-amplitude':
                    inter_score = get_interdependence_score(frequency, pulse_width, amplitude_score)
                else:
                    inter_score = get_interdependence_score(frequency, pulse_width)

                total_score = inter_score + amplitude_score

            else:
                amp_comp_pup = bi_hill_response(amplitude, ideal_amp_pup - amp_rise_pup, ideal_amp_pup + amp_fall_pup)
                amp_comp_hrv = bi_hill_response(amplitude, ideal_amp_hrv - amp_rise_hrv, ideal_amp_hrv + amp_fall_hrv)

                pup_score = 100 * amp_comp_pup 
                hrv_score = 100 * amp_comp_hrv 
                w_pup, w_hrv = 0.5, 0.5
                amplitude_score = w_pup * pup_score + w_hrv * hrv_score
                
                def get_interdependence_score(f, w, Amplitude_Scale_Factor = 1.0):
                    scores = []
                    if Amplitude_Scale_Factor != 1.0:
                        for (opt_f, opt_w) in optima:
                            dist = (((f - opt_f)**2 / (2 * freq_spread**2))* Amplitude_Scale_Factor +
                                    ((w - opt_w)**2 / (2 * width_spread**2)))
                            scores.append(np.exp(-dist))
                    else:
                        for (opt_f, opt_w) in optima:
                            dist = (((f - opt_f)**2 / (2 * freq_spread**2)) +
                                    ((w - opt_w)**2 / (2 * width_spread**2)))
                            scores.append(np.exp(-dist))

                    return max(scores)

                noise_type = patient_type.get('noise', 'normal')
                if noise_type == 'normal':
                    noise = rng.normal(0, 2.0)  # e.g., standard deviation of 2.0
                elif noise_type == 'heteroscedastic':
                        # Noise scales with signal magnitude
                    noise = rng.normal(0, 0.05 * amplitude_score)
                elif noise_type == 'heavy_tailed':
                        # Student's t-distribution
                    noise = rng.standard_t(df=3) * 2.0
                else:
                    noise = 0.0

                if patient_type.get('unique_parameter_interaction') == 'pulsewidth-amplitude':
                    inter_score = get_interdependence_score(frequency, pulse_width, amplitude_score)
                else:
                    inter_score = get_interdependence_score(frequency, pulse_width)

        
                total_score = inter_score + amplitude_score + noise
        
            return total_score
        
        amplitude_grid = np.linspace(0.5, 5.0, 10000)
        amplitude_scores = realistic_patient_curve(amplitude_grid, 10, 500)
        max_possible_score = np.max(amplitude_scores)
        highest_amplitude = amplitude_grid[np.argmax(amplitude_scores)]


        # --- Bayesian Optimization ---
        
        optimizer_multi = BayesianOptimization(
            f=realistic_patient_curve,
            pbounds=search_boundaries,
            acquisition_function=ExpectedImprovement(xi=0.01),
            random_state=patient_and_machine_startSeed,
            verbose=0
        )
        print(f"\n--- Starting Bayesian Optimization ({num_simulation_trials} iterations) ---")
        bo_start_time = time.perf_counter()
        optimizer_multi.maximize(init_points=max(1, num_simulation_trials // 10), n_iter=num_simulation_trials - max(1, num_simulation_trials // 10))

        bo_elapsed_seconds = time.perf_counter() - bo_start_time
        print(f"Bayesian Optimization completed in {bo_elapsed_seconds:.3f} seconds.")

        
        # --- TPE Optimization ---
        tpe_start_time = time.perf_counter()
        def objective_tpe(params):
                score = realistic_patient_curve(params['amplitude'], params['frequency'], params['pulse_width'])
                return {'loss': -score, 'status': STATUS_OK}
        
        space = {
            'amplitude': hp.uniform('amplitude', search_boundaries['amplitude'][0], search_boundaries['amplitude'][1]),
            'frequency': hp.uniform('frequency', search_boundaries['frequency'][0], search_boundaries['frequency'][1]),
            'pulse_width': hp.uniform('pulse_width', search_boundaries['pulse_width'][0], search_boundaries['pulse_width'][1])
        }
        trials = Trials()
        print(f"\n--- Starting TPE Optimization ({num_simulation_trials} iterations) ---")
        fmin(fn=objective_tpe, space=space, algo=tpe.suggest, max_evals=num_simulation_trials, trials=trials, rstate=np.random.default_rng(patient_and_machine_startSeed))
        tpe_elapsed_seconds = time.perf_counter() - tpe_start_time
        print(f"TPE completed in {tpe_elapsed_seconds:.3f} seconds.")
        
        # --- SMAC Optimization ---
        smac_start_time = time.perf_counter()
        configspace = ConfigurationSpace(seed=patient_and_machine_startSeed)
        configspace.add([
            Float('amplitude', bounds=search_boundaries['amplitude']),
            Float('frequency', bounds=search_boundaries['frequency']),
            Float('pulse_width', bounds=search_boundaries['pulse_width']),
        ])
        
        def objective_smac(config, seed=patient_and_machine_startSeed):
            return -realistic_patient_curve(
                float(config['amplitude']),
                float(config['frequency']),
                float(config['pulse_width']),
            )
        
        print(f"\n--- Starting SMAC Optimization ({num_simulation_trials} iterations) ---")
        scenario = Scenario(
            configspace,
            deterministic=True,  # no measurement noise is added to realistic_patient_curve
            n_trials=num_simulation_trials,
            seed=patient_and_machine_startSeed,
        )
        smac = HyperparameterOptimizationFacade(scenario, objective_smac)
        incumbent = smac.optimize()
        smac_score = -smac.runhistory.get_cost(incumbent)

        smac_elapsed_seconds = time.perf_counter() - smac_start_time
        print(f"SMAC completed in {smac_elapsed_seconds:.3f} seconds.")

        
        # --- Random Search ---
        rs_start_time = time.perf_counter()
        random_search_rng = np.random.default_rng(patient_and_machine_startSeed)
        best_random_score = -np.inf
        best_random_params = {}
        random_search_scores = []  # score at each evaluation, in order, for convergence tracking
        print(f"\n--- Starting Random Search Optimization ({num_simulation_trials} iterations) ---")
        for _ in range(num_simulation_trials):
            a, f, w = [random_search_rng.uniform(search_boundaries[k][0], search_boundaries[k][1]) for k in ['amplitude', 'frequency', 'pulse_width']]
            current_score = realistic_patient_curve(a, f, w)
            random_search_scores.append(current_score)
            if current_score > best_random_score:
                best_random_score = current_score
                best_random_params = {'amplitude': a, 'frequency': f, 'pulse_width': w}

        rs_elapsed_seconds = time.perf_counter() - rs_start_time
        print(f"Random Search completed in {rs_elapsed_seconds:.3f} seconds.")

        comparison_data = {
            'Method': ['Bayesian Optimization', 'TPE Optimization', 'SMAC Optimization', 'Random Search'],
            'Surface Family': [patient_profile['surface_family']] * 4,
            'Seed': [patient_and_machine_startSeed] * 4,
            'Best Possible Score': [max_possible_score] * 4,
            'Best Computed Score': [optimizer_multi.max['target'], -trials.best_trial['result']['loss'], smac_score, best_random_score],
            
            #Adding because 2 negatives make a plus sign in TPE optimizaiton.
            'Simple Regret': [max_possible_score - optimizer_multi.max['target'], max_possible_score + trials.best_trial['result']['loss'], max_possible_score - smac_score, max_possible_score-best_random_score],
            'Normalized Regret': [(max_possible_score - optimizer_multi.max['target'])/max_possible_score, 
                                    (max_possible_score + trials.best_trial['result']['loss'])/max_possible_score,
                                    (max_possible_score - smac_score)/max_possible_score,
                                    (max_possible_score-best_random_score)/max_possible_score],

            'Optimal Amplitude (mA)': [optimizer_multi.max['params']['amplitude'], trials.argmin['amplitude'], float(incumbent['amplitude']), best_random_params['amplitude']],
            'Optimal Frequency (Hz)': [optimizer_multi.max['params']['frequency'], trials.argmin['frequency'], float(incumbent['frequency']), best_random_params['frequency']],
            'Optimal Pulse Width (us)': [optimizer_multi.max['params']['pulse_width'], trials.argmin['pulse_width'], float(incumbent['pulse_width']), best_random_params['pulse_width']],
            'Sec To Compute': [bo_elapsed_seconds, tpe_elapsed_seconds, smac_elapsed_seconds, rs_elapsed_seconds]
        }

        all_patients_results.append(pd.DataFrame(comparison_data))

        patient_and_machine_startSeed += 1

    return all_patients_results







ModuleNotFoundError: No module named 'pandas'

In [ ]:
num_trials_depth = 40
num_runs = 40

all_comparison_results = run_optimization_comparison(
    num_simulation_trials=num_trials_depth,
    amount_of_patients=num_runs,
    patient_and_machine_startSeed=10_000,
    patient_type=baseline_family,
)

final_comparison_df = pd.concat(
    all_comparison_results,
    ignore_index=True,
)

print("\n--- Combined Optimization Comparison Results ---")
display(final_comparison_df.head())
final_comparison_df.to_csv('combined_optimization_results.csv', index=False)

### Visualizing the 3D Interaction Landscape
This plot illustrates the two 'Gaussian hills' created by the `inter_score` calculation, representing the optimal (Frequency, Pulse Width) pairs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Define parameters based on your model
optima = [(10.0, 500.0), (25.0, 100.0)]
freq_spread, width_spread = 5.0, 100.0

# Create a grid of Frequency and Pulse Width values
freq_range = np.linspace(5, 50, 100)
width_range = np.linspace(50, 600, 100)
F, W = np.meshgrid(freq_range, width_range)

# Calculate inter_score for the grid
def get_score_grid(f, w):
    scores = []
    for (opt_f, opt_w) in optima:
        dist = (((f - opt_f)**2 / (2 * freq_spread**2)) +
                ((w - opt_w)**2 / (2 * width_spread**2)))
        scores.append(np.exp(-dist))
    return np.maximum(scores[0], scores[1])

Z = get_score_grid(F, W)


fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(F, W, Z, cmap='viridis', edgecolor='none', alpha=0.8)
ax.view_init(elev=25, azim=40)
ax.dist = 12
ax.set_zlim(0, 1.1)


ax.set_xlabel('Frequency (Hz)', labelpad=15)
ax.set_ylabel('Pulse Width (us)', labelpad=15)

ax.text2D(-0.06, 0.5, "Interdependence Score", transform=ax.transAxes,
          rotation=90, fontsize=12, fontweight='normal', va='center')

ax.set_title('Parameter Interdependence', pad=15)

# Add colorbar
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10)

plt.subplots_adjust(left=0.15, right=0.9, top=0.9, bottom=0.1)
plt.show()

Calculates the theoretical max of the combined scores.

In [ ]:
def calculate_theoretical_max():
    # Constants from your realistic_patient_curve function
    ideal_amp_pup = 4.4
    amp_spread_pup = 0.5
    ideal_amp_hrv = 4.0
    amp_spread_hrv = 0.4
    w_pup, w_hrv = 0.5, 0.5

    # Range of amplitudes to test
    amps = np.linspace(3.0, 5.0, 10000)

    # Calculate scores assuming inter_score = 1.0 (perfect freq/width match)
    pup_scores = 100 * np.exp(-((amps - ideal_amp_pup)**2) / (2 * amp_spread_pup**2))
    hrv_scores = 100 * np.exp(-((amps - ideal_amp_hrv)**2) / (2 * amp_spread_hrv**2))

    combined_scores = w_pup * pup_scores + w_hrv * hrv_scores

    max_idx = np.argmax(combined_scores)
    return combined_scores[max_idx], amps[max_idx]

theoretical_max, best_amp = calculate_theoretical_max()
print(f"Theoretical Maximum Combined Score: {theoretical_max:.2f}%")
print(f"Optimal Amplitude for this peak: {best_amp:.3f} mA")

Theoretical Maximum Combined Score: 90.73%
Optimal Amplitude for this peak: 4.151 mA


## Visualization of Optimization Comparison Results

This graph compares the performance and consistency of Bayesian Optimization, TPE, and SMAC based on their 'Best Combined Score' across multiple runs.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.boxplot(x='Method', y='Best Combined Score', data=final_comparison_df, palette='viridis', hue='Method', legend=False)
plt.title('Distribution of Best Combined Score by Optimization Method (Across Multiple Runs)')
plt.xlabel('Optimization Method')
plt.ylabel('Best Combined Score')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## Statistics about Optimization Methods


In [ ]:
import pandas as pd

# Group by 'Method' and calculate descriptive statistics for 'Best Combined Score'
score_stats = final_comparison_df.groupby('Method')['Best Combined Score'].agg(['mean', 'std', 'min', 'max']).reset_index()
print("\n--- Statistics for Best Combined Score ---")
display(score_stats)

# Calculate descriptive statistics for each optimal parameter
parameters = ['Optimal Amplitude (mA)', 'Optimal Frequency (Hz)', 'Optimal Pulse Width (us)']

all_param_stats = []
for param in parameters:
    param_stats = final_comparison_df.groupby('Method')[param].agg(['mean', 'std', 'min', 'max']).reset_index()
    param_stats['Parameter'] = param
    all_param_stats.append(param_stats)

# Concatenate all parameter statistics into a single DataFrame
final_param_stats_df = pd.concat(all_param_stats, ignore_index=True)

print("\n--- Statistics for Optimal Parameters ---")
display(final_param_stats_df)


## ANOVA Test for Optimization Methods


In [ ]:
from scipy import stats

# Extract the 'Best Combined Score' for each optimization method
bayesian_scores = final_comparison_df[final_comparison_df['Method'] == 'Bayesian Optimization']['Best Combined Score']
tpe_scores = final_comparison_df[final_comparison_df['Method'] == 'TPE Optimization']['Best Combined Score']
smac_scores = final_comparison_df[final_comparison_df['Method'] == 'SMAC Optimization']['Best Combined Score']
random_search_scores = final_comparison_df[final_comparison_df['Method'] == 'Random Search']['Best Combined Score']

# Perform one-way ANOVA including Random Search
f_statistic, p_value = stats.f_oneway(bayesian_scores, tpe_scores, smac_scores, random_search_scores)

print(f"--- ANOVA Results for Best Combined Score ---")
print(f"F-statistic: {f_statistic:.5f}")
print(f"P-value: {p_value:.5f}")

alpha = 0.05 # Significance level
if p_value < alpha:
    print(f"Conclusion: Reject the null hypothesis. There is a statistically significant difference (p < {alpha}) in the mean 'Best Combined Score' among the optimization methods.")
else:
    print(f"Conclusion: Fail to reject the null hypothesis. There is no statistically significant difference (p >= {alpha}) in the mean 'Best Combined Score' among the optimization methods.")

--- ANOVA Results for Best Combined Score ---
F-statistic: 7.59857
P-value: 0.00009
Conclusion: Reject the null hypothesis. There is a statistically significant difference (p < 0.05) in the mean 'Best Combined Score' among the optimization methods.


## Post-Hoc Analysis: Tukey's HSD Test

Since ANOVA indicates a statistically significant difference among the optimization methods, a post-hoc test is done to identify  which pairs of methods have statistically different mean Best Combined Score.

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Combine all scores into a single array and create corresponding group labels
all_scores = pd.concat([bayesian_scores, tpe_scores, smac_scores, random_search_scores])
groups = ['Bayesian Optimization'] * len(bayesian_scores) + \
         ['TPE Optimization'] * len(tpe_scores) + \
         ['SMAC Optimization'] * len(smac_scores) + \
         ['Random Search'] * len(random_search_scores)

# Perform Tukey's HSD post-hoc test
tukey_result = pairwise_tukeyhsd(endog=all_scores, groups=groups, alpha=0.05)

print("--- Tukey's HSD Post-Hoc Test Results ---")
print(tukey_result)

# Results visualized
tukey_result.plot_simultaneous()
plt.title("Tukey's HSD - Best Combined Score")
plt.show()